# NYC One-Bed Rents — Central Tendency (Mean / Median / Mode) — Solution

**Short name (GitHub):** `HousingCT_Py`
**Lab source:** Codecademy *Stats with Python* → *Central Tendency for Housing Data* (StreetEasy Brooklyn / Manhattan / Queens).
**Data:** `data/brooklyn-one-bed.csv`, `data/manhattan-one-bed.csv`, `data/queens-one-bed.csv`.
**Companion files:** `HousingCT_Py_Solution.ipynb`, `HousingCT_Py_Reusable_Template.ipynb`, `HousingCT_Py_Cheatsheet.docx`, `HousingCT_Py_Project_Memo.docx`, `HousingCT_Py_Strategy_Guide.docx`, `HousingCT_Py_1Page_Summary_Report.docx`, `housingct_py_flowchart.png`.

This notebook is the worked key. Use `HousingCT_Py_Practice_Skeleton.ipynb` to practice first.

**Flowchart of the desired outcome:**

![Flowchart](housingct_py_flowchart.png)

---
## Learning objectives
1. Load three CSVs and extract numeric rent vectors.
2. Compute the **mean** by hand and with NumPy / pandas.
3. Compute the **median** (sort + built-in).
4. Compute the **mode** with SciPy and with `value_counts`.
5. Compare boroughs and diagnose right-skew (mean vs median vs mode).
6. Overlay the three statistics on histograms.
7. Write short interpretations for analyst / technician / executive / nonspecialist audiences.
8. Run alternate implementations, extra practice, and a Monte-Carlo simulation with editable knobs.



## Inline cheat-sheet (keep this cell visible)

See also **`HousingCT_Py_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Load | `df = pd.read_csv("data/brooklyn-one-bed.csv")` |
| Rent vector | `price = df["rent"]`  (Series) or `df["rent"].to_numpy()` |
| Mean | \(\bar x = \sum x_i / n\) → `np.mean(x)` or `x.mean()` |
| Median (odd n) | middle of sorted `x` |
| Median (even n) | average of the two middle values → `np.median` handles both |
| Mode | most frequent value → `stats.mode(x, keepdims=True)` or `x.value_counts().idxmax()` |
| Count of mode | `stats.mode(...).count[0]` or `x.value_counts().iloc[0]` |
| Skew rule of thumb | right-skew rents: **mode ≤ median ≤ mean** |
| Overlay lines | `ax.axvline(mean, ...)` |
| Fair 1-bed slice | `df.loc[df["bedrooms"] == 1, "rent"]` |

**Data note:** despite the filename, `brooklyn-one-bed.csv` is a mixed-bedroom StreetEasy extract (studios through 5-beds). Manhattan and Queens files are already 1-bed only. Compute the official lesson numbers on the raw `rent` column first, then filter Brooklyn in *More practice*.



## 0. Packages


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from collections import Counter

pd.set_option("display.max_columns", 12)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


## 1. Load the three StreetEasy extracts


In [ ]:
brooklyn_one_bed = pd.read_csv("data/brooklyn-one-bed.csv")
manhattan_one_bed = pd.read_csv("data/manhattan-one-bed.csv")
queens_one_bed = pd.read_csv("data/queens-one-bed.csv")

brooklyn_price = brooklyn_one_bed["rent"]
manhattan_price = manhattan_one_bed["rent"]
queens_price = queens_one_bed["rent"]

print("Brooklyn columns:", list(brooklyn_one_bed.columns))
print(brooklyn_one_bed.head(3))
print("\nManhattan extra columns:",
      sorted(set(manhattan_one_bed.columns) - set(brooklyn_one_bed.columns)))


## 2. Observe the data


In [ ]:
def peek(name, df):
    print(f"=== {name} ===")
    print("n =", len(df), "  rent min/max =", df["rent"].min(), df["rent"].max())
    if "bedrooms" in df.columns:
        print("bedrooms:\n", df["bedrooms"].value_counts().sort_index().to_string())
    print()

peek("Brooklyn", brooklyn_one_bed)
peek("Manhattan", manhattan_one_bed)
peek("Queens", queens_one_bed)
print("NOTE: Brooklyn is mixed-bedroom; Manhattan and Queens are 1-bed only.")


## 3. Find the mean


In [ ]:
# manual sanity check on the first four Brooklyn rents
first4 = brooklyn_price.iloc[:4]
print("manual first-4 mean:", first4.sum() / 4, " vs ", first4.mean())

brooklyn_mean = float(np.mean(brooklyn_price))
manhattan_mean = float(np.mean(manhattan_price))
queens_mean = float(np.mean(queens_price))

print(f"The mean price in Brooklyn is {brooklyn_mean:.2f}")
print(f"The mean price in Manhattan is {manhattan_mean:.2f}")
print(f"The mean price in Queens is {queens_mean:.2f}")


## 4. Find the median


In [ ]:
# manual median on a tiny odd-n slice
demo = np.sort(brooklyn_price.iloc[:5].to_numpy())
print("sorted first-5:", demo, " manual median:", demo[2])

brooklyn_median = float(np.median(brooklyn_price))
manhattan_median = float(np.median(manhattan_price))
queens_median = float(np.median(queens_price))

print("The median price in Brooklyn is", brooklyn_median)
print("The median price in Manhattan is", manhattan_median)
print("The median price in Queens is", queens_median)


## 5. Find the mode


In [ ]:
brooklyn_mode = stats.mode(brooklyn_price, keepdims=True)
manhattan_mode = stats.mode(manhattan_price, keepdims=True)
queens_mode = stats.mode(queens_price, keepdims=True)

def report_mode(name, mode_res, n):
    val = int(mode_res.mode[0])
    cnt = int(mode_res.count[0])
    print(f"The mode price in {name} is {val} and it appears {cnt} times out of {n}")

report_mode("Brooklyn", brooklyn_mode, len(brooklyn_price))
report_mode("Manhattan", manhattan_mode, len(manhattan_price))
report_mode("Queens", queens_mode, len(queens_price))


## 6. What does the data tell us?

**Lesson numbers (raw `rent` column, matching Codecademy print statements):**

| Borough | n | Mean | Median | Mode (count) |
|---------|---|------|--------|--------------|
| Queens | 232 | $2,346.25 | $2,200 | $1,750 (11) |
| Brooklyn | 1,013 | $3,327.40 | $3,000 | $2,500 (26) |
| Manhattan | 1,476 | $3,993.48 | $3,800 | $3,500 (56) |

Ranking is the same on every measure: **Manhattan > Brooklyn > Queens**.

All three boroughs are **right-skewed** (`mean > median > mode`). A handful of luxury listings pull the average above what a typical renter will actually see on the market. For "what does a 1-bed cost around here?" lead with the **median**; use the mean only when you are adding rents (portfolio / aggregate rent roll).

Queens has the tightest spread and the smallest sample — treat its mode as a soft signal, not a law of nature.



In [ ]:
rows = [
    ("Queens", queens_price, queens_mean, queens_median, queens_mode),
    ("Brooklyn", brooklyn_price, brooklyn_mean, brooklyn_median, brooklyn_mode),
    ("Manhattan", manhattan_price, manhattan_mean, manhattan_median, manhattan_mode),
]
print(f"{'Borough':<12} {'n':>5} {'mean':>10} {'median':>10} {'mode':>8} {'count':>6} {'mean-med':>10}")
for name, x, mu, med, mo in rows:
    print(f"{name:<12} {len(x):>5} {mu:10.2f} {med:10.1f} {int(mo.mode[0]):8d} {int(mo.count[0]):6d} {mu-med:10.1f}")


## 7. Assumptions

1. **Listed ask ≠ signed lease.** StreetEasy prices are advertised rents; negotiated leases can be lower (or, in a tight market, higher).
2. **The Brooklyn file is not a 1-bed file.** Ranking "1-bed cost of living" on the raw Brooklyn column mixes studios and 3-beds into the same average. Filter `bedrooms == 1` before you brief a client on 1-beds (Section 10a).
3. **Samples are not the same size or design.** Manhattan n=1,476 vs Queens n=232. Queens estimates bounce more.
4. **Coverage is "what got listed on StreetEasy,"** not the full housing stock (rent-stabilized walk-ups and informal sublets are under-counted).
5. **"Typical" is a choice.** Mean for totals, median for a typical listing, mode for the most common asking price.
6. **No inflation / season adjustment.** Cross-section snapshot, not a time series.



In [ ]:
assumptions = [
    "Listed ask ≠ signed lease.",
    "Brooklyn CSV is mixed-bedroom despite the filename.",
    "n is unbalanced (Queens 232 vs Manhattan 1,476).",
    "StreetEasy coverage ≠ full housing stock.",
    "Mean / median / mode answer different questions.",
    "Snapshot, not inflation-adjusted.",
]
for a in assumptions:
    print("-", a)


## 8. Histograms with mean / median / mode overlays


In [ ]:
bundle = [
    ("Brooklyn", brooklyn_price, "#2E86AB"),
    ("Manhattan", manhattan_price, "#E94F37"),
    ("Queens", queens_price, "#44AF69"),
]
fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.2))
for ax, (name, x, color) in zip(axes, bundle):
    mu, med = float(np.mean(x)), float(np.median(x))
    mo = int(stats.mode(x, keepdims=True).mode[0])
    xmax = np.percentile(x, 99)
    ax.hist(x[x <= xmax], bins=20, color=color, edgecolor="white", alpha=0.85)
    ax.axvline(mu, color="#111", ls="-", lw=1.6, label=f"mean ${mu:,.0f}")
    ax.axvline(med, color="#F4A261", ls="--", lw=1.8, label=f"median ${med:,.0f}")
    ax.axvline(mo, color="#7B2D8E", ls=":", lw=2.0, label=f"mode ${mo:,.0f}")
    ax.set_title(f"{name}  n={len(x):,}")
    ax.set_xlabel("Monthly rent (USD)")
    ax.legend(fontsize=7, loc="upper right")
axes[0].set_ylabel("Listings")
fig.suptitle("Rent distributions with mean / median / mode", y=1.02)
fig.tight_layout()
plt.show()
print("Reference image shipped with the project: housingct_py_hist.png")


## 9. Alternate code — same numbers, different APIs


In [ ]:
def alt_stats(x):
    x = pd.Series(x)
    mean_a = float(np.average(x))
    n = len(x)
    sorted_x = np.sort(x.to_numpy())
    if n % 2 == 1:
        med_a = float(sorted_x[n // 2])
    else:
        med_a = float(sorted_x[n//2 - 1] + sorted_x[n//2]) / 2
    med_b = float(np.quantile(x, 0.5))
    mode_a = int(x.value_counts().idxmax())
    mode_b = int(Counter(x).most_common(1)[0][0])
    return {
        "mean_avg": mean_a,
        "mean_manual": float(np.sum(x) / n),
        "median_sorted": med_a,
        "median_q50": med_b,
        "mode_vc": mode_a,
        "mode_counter": mode_b,
    }

for name, x, mu, med, mo in [
    ("Brooklyn", brooklyn_price, brooklyn_mean, brooklyn_median, brooklyn_mode),
    ("Manhattan", manhattan_price, manhattan_mean, manhattan_median, manhattan_mode),
    ("Queens", queens_price, queens_mean, queens_median, queens_mode),
]:
    a = alt_stats(x)
    print(name)
    print("  mean   lesson", round(mu, 4), " alts", a["mean_avg"], a["mean_manual"])
    print("  median lesson", med, " alts", a["median_sorted"], a["median_q50"])
    print("  mode   lesson", int(mo.mode[0]), " alts", a["mode_vc"], a["mode_counter"])


## 10. More practice


In [ ]:
# 10a Fair 1-bed card
bk_1bed = brooklyn_one_bed.loc[brooklyn_one_bed["bedrooms"] == 1, "rent"]
print("10a Brooklyn 1-bed only  n =", len(bk_1bed))
print("    mean   {:.2f}   (mixed-file mean was {:.2f})".format(bk_1bed.mean(), brooklyn_mean))
print("    median {:.0f}   (mixed-file median was {:.0f})".format(bk_1bed.median(), brooklyn_median))
print("    mode   {}     (mixed-file mode was {})".format(
    int(bk_1bed.value_counts().idxmax()), int(brooklyn_mode.mode[0])))
print("    → filtering drops the Brooklyn mean by about ${:.0f} and the '1-bed gap' vs Manhattan widens.\n".format(
    brooklyn_mean - bk_1bed.mean()))

# 10b IQR and % above the mean
print("10b IQR and share above the mean")
for name, x, mu in [("Brooklyn", brooklyn_price, brooklyn_mean),
                    ("Manhattan", manhattan_price, manhattan_mean),
                    ("Queens", queens_price, queens_mean)]:
    q1, q3 = np.percentile(x, 25), np.percentile(x, 75)
    share = (x > mu).mean()
    print(f"    {name:<10} Q1={q1:.0f}  Q3={q3:.0f}  IQR={q3-q1:.0f}  P(rent>mean)={share:.1%}")

# 10c Manhattan Village vs Harlem
print("\n10c Manhattan neighborhood slice")
nb = manhattan_one_bed["neighborhood"].fillna("")
vil = manhattan_one_bed.loc[nb.str.contains("Village", case=False), "rent"]
har = manhattan_one_bed.loc[nb.str.contains("Harlem", case=False), "rent"]
print(f"    Village-like n={len(vil)} median=${vil.median():.0f} mean=${vil.mean():.0f}")
print(f"    Harlem-like  n={len(har)} median=${har.median():.0f} mean=${har.mean():.0f}")

# 10d Brooklyn doorman premium
print("\n10d Brooklyn doorman premium (median)")
d1 = brooklyn_one_bed.loc[brooklyn_one_bed["has_doorman"] == 1, "rent"]
d0 = brooklyn_one_bed.loc[brooklyn_one_bed["has_doorman"] == 0, "rent"]
print(f"    doorman n={len(d1)} median=${d1.median():.0f}")
print(f"    no door n={len(d0)} median=${d0.median():.0f}")
print(f"    gap ${d1.median() - d0.median():.0f}")


## 11. Simulation — editable knobs


In [ ]:
# ---- knobs (edit these) ----
SEED = 42
N_REP = 400
N_SAMPLE = 200
N_OUTLIERS = 5
OUTLIER_RENT = 25000

rng = np.random.default_rng(SEED)
base = manhattan_price.to_numpy(float)

# Experiment 1 — bootstrap width of mean and median
boot_mean = np.empty(N_REP)
boot_med = np.empty(N_REP)
for i in range(N_REP):
    s = rng.choice(base, size=N_SAMPLE, replace=True)
    boot_mean[i] = s.mean()
    boot_med[i] = np.median(s)

def width(a):
    return np.percentile(a, 97.5) - np.percentile(a, 2.5)

print(f"Experiment 1  n={N_SAMPLE}  reps={N_REP}")
print(f"  mean   center ${boot_mean.mean():,.0f}   95% width ${width(boot_mean):,.0f}")
print(f"  median center ${boot_med.mean():,.0f}   95% width ${width(boot_med):,.0f}")

# sweep a few n values for a quick curve
print("  n sweep (mean width):")
for n in (30, 80, 200, 500, len(base)):
    ws = [rng.choice(base, size=n, replace=True).mean() for _ in range(N_REP)]
    print(f"    n={n:4d}  width=${width(np.array(ws)):,.0f}")

# Experiment 2 — outlier injection
sample = rng.choice(base, size=400, replace=True)
clean_mean, clean_med = sample.mean(), np.median(sample)
injected = np.concatenate([sample, np.full(N_OUTLIERS, float(OUTLIER_RENT))])
print(f"\nExperiment 2  append {N_OUTLIERS} listings at ${OUTLIER_RENT:,}")
print(f"  mean   ${clean_mean:,.0f} → ${injected.mean():,.0f}   Δ ${injected.mean()-clean_mean:,.0f}")
print(f"  median ${clean_med:,.0f} → ${np.median(injected):,.0f}   Δ ${np.median(injected)-clean_med:,.0f}")
print("Reference image: housingct_py_simulation.png")


## 12. Audience rewrite

Headline number: **Manhattan median listed 1-bed = \$3,800 (n = 1,476).**

| Audience | Sentence |
|----------|----------|
| Analyst / expert | Across 1,476 StreetEasy Manhattan 1-beds the median ask is \$3,800, the mean \$3,993 (right-skew of \$193), and the modal ask \$3,500 (56 listings). Lead with the median; the mean is reserved for aggregating a rent roll. Caveat: advertised asks, not executed leases. |
| Technician (broker / ops) | Pull `manhattan-one-bed.csv`, take column `rent`, run `median`. Confirm every row has `bedrooms == 1` (it does). Re-run monthly; do not blend in the Brooklyn file without filtering `bedrooms`. |
| Executive | A typical listed Manhattan 1-bed is about \$3,800. That is ~\$800 above Brooklyn's listed median and ~\$1,600 above Queens. Luxury tail listings pull the *average* a couple of hundred dollars higher — do not budget the average as if it were typical. |
| Nonspecialist | Half of the Manhattan one-bedrooms we looked at are listed at \$3,800 a month or less. A few very expensive apartments make the "average" look closer to \$4,000. Queens listings in the same snapshot cluster nearer \$2,200. These are asking prices, not a promise of what you will pay. |



In [ ]:
print("Manhattan median ${:,.0f}  n={}".format(manhattan_median, len(manhattan_price)))
print("Four audience drafts live in the markdown cell above.")


## 13. Recap

| Measure | Lesson API | Typical use on this file |
|---------|------------|--------------------------|
| Mean | `np.mean` | Portfolio / aggregate rent |
| Median | `np.median` | "Typical listed 1-bed" |
| Mode | `stats.mode` | Most common asking price |

Official raw-column results: Brooklyn \$3,327 / \$3,000 / \$2,500; Manhattan \$3,993 / \$3,800 / \$3,500; Queens \$2,346 / \$2,200 / \$1,750.

After filtering Brooklyn to true 1-beds the mean falls to ~\$2,785 and the median sits near \$2,800 — the mixed-file Brooklyn average was being lifted by larger units.

**Not rental advice. Not a valuation.**

